In [0]:
import pandas as pd
from pyspark.sql.functions import current_timestamp

# TU MISMO LINK - pandas lo puede leer directo sin requests
github_raw_url = "https://raw.githubusercontent.com/lynxiondev/iot-telemetry-data-platform/main/data/raw/raw_telemetry_dirty.csv"

print("⏳ Leyendo CSV desde GitHub Raw (vía Pandas, necesario en Serverless)...")
# pandas lee la url directo, no hace falta io.StringIO ni requests
df_pandas = pd.read_csv(github_raw_url)
print(f"✅ Leído en memoria: {len(df_pandas)} filas")

# 2. A Spark
df_bronze = spark.createDataFrame(df_pandas)

# 3. Columnas de auditoría pro - esto es lo que te faltaba para que se vea senior
df_bronze_final = df_bronze \
    .withColumn("ingest_timestamp", current_timestamp())

# 4. Guardamos como vos ya hacías
table_name = "bronze_telemetry"

df_bronze_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✅ ¡ÉXITO! Guardado en tabla gestionada: {table_name}")
display(spark.table(table_name).limit(5))

In [0]:
# Consultar el historial de versiones de nuestra tabla
display(spark.sql("DESCRIBE HISTORY bronze_telemetry"))

In [0]:
# Viajar en el tiempo - ver cómo estaba antes del último overwrite
# display(spark.sql("SELECT * FROM bronze_telemetry VERSION AS OF 0 LIMIT 5"))